# Customer Credit Risk Dataset: Complete Preprocessing & Feature Engineering Workflow

This notebook performs the full preprocessing workflow on `customer_credit_risk_practice_dataset.csv`. Each task is separated into an intuition cell and a code cell so the purpose of every step is clear.

## 1. Import Required Libraries

**Intuition:** We import data handling, visualization, preprocessing, transformation, and pipeline tools first so every later step can run without repeated imports.

In [61]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    MinMaxScaler,
    MaxAbsScaler,
    RobustScaler,
    Normalizer,
    StandardScaler,
    Binarizer,
    KBinsDiscretizer,
    FunctionTransformer,
    PowerTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Load the Dataset

**Intuition:** We load the raw CSV and keep an untouched copy. This is useful because preprocessing changes data, and we may need the original values for comparison.

In [62]:
file_path = 'customer_credit_risk_practice_dataset.csv'
df = pd.read_csv(file_path, na_values=['null'])
raw_df = df.copy()

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (80, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,CUST0001,34,Male,North,Bachelor,Salaried,62000.0,15000.0,Home,710.0,0.0,42.0,24.8,2021-03-14,0
1,CUST0002,28,Female,South,Master,Self-Employed,88000.0,22000.0,Car,735.0,1.0,67.0,31.5,2020-11-02,0
2,CUST0003,45,Male,East,High School,Salaried,54000.0,12000.0,Education,668.0,2.0,36.0,28.2,2019-06-25,0
3,CUST0004,39,Female,West,Bachelor,Unemployed,NaN,18000.0,Personal,602.0,4.0,21.0,44.7,2022-01-10,1
4,CUST0005,52,Male,North,PhD,Salaried,145000.0,40000.0,Home,805.0,0.0,89.0,19.4,2018-08-19,0


## 3. Understand Dataset Structure

**Intuition:** Before cleaning, we inspect columns, data types, missing values, and basic statistics. This tells us which preprocessing methods are required.

In [63]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes)

print('\nMissing values per column:')
display(df.isna().sum())

print('\nNumerical summary:')
display(df.describe())

print('\nCategorical summary:')
display(df.describe(include='object'))

Columns:
['customer_id', 'age', 'gender', 'region', 'education_level', 'employment_type', 'annual_income', 'loan_amount', 'loan_purpose', 'credit_score', 'repayment_history', 'transaction_count', 'spending_ratio', 'join_date', 'default_flag']

Data types:


customer_id           object
age                    int64
gender                object
region                object
education_level       object
employment_type       object
annual_income        float64
loan_amount          float64
loan_purpose          object
credit_score         float64
repayment_history    float64
transaction_count    float64
spending_ratio       float64
join_date             object
default_flag           int64
dtype: object


Missing values per column:


customer_id          0
age                  0
gender               2
region               0
education_level      0
employment_type      1
annual_income        4
loan_amount          3
loan_purpose         0
credit_score         3
repayment_history    2
transaction_count    2
spending_ratio       3
join_date            2
default_flag         0
dtype: int64


Numerical summary:


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,default_flag
count,80.00000,76.000000,77.000000,77.000000,78.000000,78.000000,77.000000,80.000000
mean,39.73750,80236.842105,22874.025974,686.116883,2.128205,53.705128,35.164935,0.237500
std,11.01523,46104.480887,16093.700979,73.521922,1.995833,27.961062,13.946314,0.428236
min,22.00000,18000.000000,5000.000000,540.000000,0.000000,9.000000,14.200000,0.000000
25%,30.75000,51250.000000,12000.000000,630.000000,1.000000,34.250000,24.900000,0.000000
50%,39.00000,74000.000000,18000.000000,692.000000,2.000000,52.500000,31.500000,0.000000
75%,47.25000,97250.000000,28000.000000,735.000000,3.000000,70.750000,42.100000,0.000000
max,64.00000,250000.000000,90000.000000,840.000000,8.000000,130.000000,72.800000,1.000000



Categorical summary:


,customer_id,gender,region,education_level,employment_type,loan_purpose,join_date
count,80,78,80,80,79,80,78
unique,80,2,4,4,5,6,78
top,CUST0001,Male,North,Bachelor,Salaried,Home,2021-03-14
freq,1,39,20,34,42,17,1


## 4. Convert Date Column

**Intuition:** `join_date` is stored as text after CSV loading. Converting it to datetime allows us to extract year, month, weekday, and customer tenure.

In [64]:
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')

print('join_date converted to datetime.')
display(df[['customer_id', 'join_date']].head())

join_date converted to datetime.


,customer_id,join_date
0,CUST0001,2021-03-14
1,CUST0002,2020-11-02
2,CUST0003,2019-06-25
3,CUST0004,2022-01-10
4,CUST0005,2018-08-19


## 5. Separate Column Groups

**Intuition:** Different column types need different treatments. IDs are identifiers, categorical features need encoding, numeric features need imputation/scaling, and the target must be kept separate.

In [65]:
id_col = 'customer_id'
target_col = 'default_flag'
date_col = 'join_date'

ordinal_cols = ['education_level']
nominal_cols = ['gender', 'region', 'employment_type', 'loan_purpose']
numeric_cols = ['age', 'annual_income', 'loan_amount', 'credit_score', 'repayment_history', 'transaction_count', 'spending_ratio']

print('ID column:', id_col)
print('Target column:', target_col)
print('Date column:', date_col)
print('Ordinal columns:', ordinal_cols)
print('Nominal columns:', nominal_cols)
print('Numeric columns:', numeric_cols)

ID column: customer_id
Target column: default_flag
Date column: join_date
Ordinal columns: ['education_level']
Nominal columns: ['gender', 'region', 'employment_type', 'loan_purpose']
Numeric columns: ['age', 'annual_income', 'loan_amount', 'credit_score', 'repayment_history', 'transaction_count', 'spending_ratio']


## 6. Missing Value Strategy

**Intuition:** Missing values are handled according to feature type. Numerical features are imputed with median because median is resistant to outliers. Categorical features are imputed with the most frequent category. Date values are imputed using the median joining date.

In [66]:
df_clean = df.copy()

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df_clean[numeric_cols] = num_imputer.fit_transform(df_clean[numeric_cols])
df_clean[nominal_cols + ordinal_cols] = cat_imputer.fit_transform(df_clean[nominal_cols + ordinal_cols])

median_join_date = df_clean[date_col].dropna().median()
df_clean[date_col] = df_clean[date_col].fillna(median_join_date)

print('Missing values after imputation:')
display(df_clean.isna().sum())

Missing values after imputation:


customer_id          0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
loan_amount          0
loan_purpose         0
credit_score         0
repayment_history    0
transaction_count    0
spending_ratio       0
join_date            0
default_flag         0
dtype: int64

## 7. KNN Imputation Demonstration

**Intuition:** KNN imputation fills missing numerical values using similar rows. It is useful when numeric columns are related, but it is shown separately because the final cleaned dataset uses simpler and more explainable median imputation.

In [67]:
knn_demo_cols = ['annual_income', 'loan_amount', 'credit_score', 'transaction_count']
knn_imputer = KNNImputer(n_neighbors=3)
knn_imputed_demo = raw_df[knn_demo_cols].replace('null', np.nan).astype(float).copy()
knn_imputed_demo[knn_demo_cols] = knn_imputer.fit_transform(knn_imputed_demo[knn_demo_cols])

print('KNN imputation demo output:')
display(knn_imputed_demo.head())

KNN imputation demo output:


,annual_income,loan_amount,credit_score,transaction_count
0,62000.000000,15000.0,710.0,42.0
1,88000.000000,22000.0,735.0,67.0
2,54000.000000,12000.0,668.0,36.0
3,76666.666667,18000.0,602.0,21.0
4,145000.000000,40000.0,805.0,89.0


## 8. Date Feature Extraction

**Intuition:** Dates are difficult for ML models in raw form. We convert `join_date` into separate numerical features such as join year, month, day, weekday, and tenure.

In [68]:
reference_date = df_clean[date_col].max()

df_clean['join_year'] = df_clean[date_col].dt.year
df_clean['join_month'] = df_clean[date_col].dt.month
df_clean['join_day'] = df_clean[date_col].dt.day
df_clean['join_weekday'] = df_clean[date_col].dt.weekday
df_clean['customer_tenure_days'] = (reference_date - df_clean[date_col]).dt.days

date_features = ['join_year', 'join_month', 'join_day', 'join_weekday', 'customer_tenure_days']
print('Date features created:')
display(df_clean[[id_col, date_col] + date_features].head())

Date features created:


,customer_id,join_date,join_year,join_month,join_day,join_weekday,customer_tenure_days
0,CUST0001,2021-03-14,2021,3,14,6,1180
1,CUST0002,2020-11-02,2020,11,2,0,1312
2,CUST0003,2019-06-25,2019,6,25,1,1808
3,CUST0004,2022-01-10,2022,1,10,0,878
4,CUST0005,2018-08-19,2018,8,19,6,2118


## 9. Feature Construction

**Intuition:** New features can make patterns easier for ML models to learn. Debt-to-income ratio, average monthly transactions, and spending-to-income ratio represent risk and customer behavior more directly than raw values alone.

In [69]:
epsilon = 1e-6

df_clean['debt_to_income_ratio'] = df_clean['loan_amount'] / (df_clean['annual_income'] + epsilon)
df_clean['avg_monthly_transactions'] = df_clean['transaction_count'] / 6
df_clean['spending_to_income_ratio'] = (df_clean['spending_ratio'] / 100) * df_clean['annual_income']

engineered_features = ['debt_to_income_ratio', 'avg_monthly_transactions', 'spending_to_income_ratio']
print('Engineered features created:')
display(df_clean[[id_col] + engineered_features].head())

Engineered features created:


,customer_id,debt_to_income_ratio,avg_monthly_transactions,spending_to_income_ratio
0,CUST0001,0.241935,7.000000,15376.0
1,CUST0002,0.250000,11.166667,27720.0
2,CUST0003,0.222222,6.000000,15228.0
3,CUST0004,0.243243,3.500000,33078.0
4,CUST0005,0.275862,14.833333,28130.0


## 10. Outlier Detection with IQR

**Intuition:** Outliers can distort mean-based models and scaling. IQR detects values far below Q1 or far above Q3. Here we identify outliers first before handling them.

In [70]:
outlier_cols = numeric_cols + engineered_features
outlier_summary = []

for col in outlier_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_summary.append({'column': col, 'lower_bound': lower, 'upper_bound': upper, 'outlier_count': count})

outlier_summary_df = pd.DataFrame(outlier_summary)
display(outlier_summary_df)

,column,lower_bound,upper_bound,outlier_count
0,age,6.000000,72.000000,0
1,annual_income,-13250.000000,160750.000000,6
2,loan_amount,-11062.500000,51437.500000,4
3,credit_score,481.875000,886.875000,0
4,repayment_history,-2.000000,6.000000,3
5,transaction_count,-18.500000,123.500000,1
6,spending_ratio,0.825000,66.025000,3
7,debt_to_income_ratio,0.123076,0.415662,9
8,avg_monthly_transactions,-3.083333,20.583333,1
9,spending_to_income_ratio,1111.625000,45344.625000,7


## 11. Outlier Handling with Capping/Winsorization

**Intuition:** Instead of deleting rows, capping limits extreme values to acceptable bounds. This keeps all customers in the dataset while reducing the effect of extreme values.

In [71]:
df_capped = df_clean.copy()

for col in outlier_cols:
    q1 = df_capped[col].quantile(0.25)
    q3 = df_capped[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)

print('Outlier capping completed.')
display(df_capped[outlier_cols].describe())

Outlier capping completed.


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,debt_to_income_ratio,avg_monthly_transactions,spending_to_income_ratio
count,80.00000,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000,80.000000
mean,39.73750,77293.750000,21638.125000,686.337500,2.075000,53.593750,34.897188,0.276145,8.932292,23888.379688
std,11.01523,37792.315855,12589.507055,72.121196,1.847167,27.386716,13.367132,0.064657,4.564453,9865.554353
min,22.00000,18000.000000,5000.000000,540.000000,0.000000,9.000000,14.200000,0.123076,1.500000,8190.000000
25%,30.75000,52000.000000,12375.000000,633.750000,1.000000,34.750000,25.275000,0.232796,5.791667,17699.000000
50%,39.00000,74000.000000,18000.000000,692.000000,2.000000,52.500000,31.500000,0.250000,8.750000,21262.000000
75%,47.25000,95500.000000,28000.000000,735.000000,3.000000,70.250000,41.575000,0.305942,11.708333,28757.250000
max,64.00000,160750.000000,51437.500000,840.000000,6.000000,123.500000,66.025000,0.415662,20.583333,45344.625000


## 12. Ordinal Encoding

**Intuition:** `education_level` has a meaningful order, so ordinal encoding is appropriate. Higher education levels receive larger numerical values.

In [72]:
education_order = [['High School', 'Bachelor', 'Master', 'PhD']]
ordinal_encoder = OrdinalEncoder(categories=education_order)

df_encoded = df_capped.copy()
df_encoded['education_level_encoded'] = ordinal_encoder.fit_transform(df_encoded[['education_level']]).astype(int)

display(df_encoded[[id_col, 'education_level', 'education_level_encoded']].head())

,customer_id,education_level,education_level_encoded
0,CUST0001,Bachelor,1
1,CUST0002,Master,2
2,CUST0003,High School,0
3,CUST0004,Bachelor,1
4,CUST0005,PhD,3


## 13. One-Hot Encoding

**Intuition:** Nominal columns such as gender, region, employment type, and loan purpose do not have natural order. One-hot encoding prevents the model from assuming false ranking between categories.

In [73]:
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
onehot_array = onehot_encoder.fit_transform(df_encoded[nominal_cols])
onehot_feature_names = onehot_encoder.get_feature_names_out(nominal_cols)
onehot_df = pd.DataFrame(onehot_array, columns=onehot_feature_names, index=df_encoded.index)

print('One-hot encoded columns created:', len(onehot_feature_names))
display(onehot_df.head())

One-hot encoded columns created: 17


,gender_Female,gender_Male,region_East,region_North,region_South,region_West,employment_type_Retired,employment_type_Salaried,employment_type_Self-Employed,employment_type_Student,employment_type_Unemployed,loan_purpose_Business,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Personal
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 14. Label Encoding Demo for Target

**Intuition:** Label encoding is usually useful for target labels. Here `default_flag` is already binary, but this cell demonstrates the method correctly.

In [74]:
label_encoder = LabelEncoder()
df_encoded['default_flag_label_encoded'] = label_encoder.fit_transform(df_encoded[target_col])

display(df_encoded[[target_col, 'default_flag_label_encoded']].drop_duplicates().sort_values(target_col))

,default_flag,default_flag_label_encoded
0,0,0
3,1,1


## 15. Binning and Discretization

**Intuition:** Binning converts continuous numbers into groups. This can simplify patterns such as low/medium/high income or risky/safe credit score ranges.

In [75]:
df_encoded['income_bin'] = pd.cut(
    df_encoded['annual_income'],
    bins=[0, 50000, 100000, np.inf],
    labels=['Low', 'Medium', 'High']
)

df_encoded['credit_score_bin'] = pd.cut(
    df_encoded['credit_score'],
    bins=[0, 580, 670, 740, 850],
    labels=['Poor', 'Fair', 'Good', 'Excellent'],
    include_lowest=True
)

kbins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile', quantile_method='averaged_inverted_cdf')
df_encoded['transaction_count_kbin'] = kbins.fit_transform(df_encoded[['transaction_count']]).astype(int)

display(df_encoded[[id_col, 'annual_income', 'income_bin', 'credit_score', 'credit_score_bin', 'transaction_count', 'transaction_count_kbin']].head())

,customer_id,annual_income,income_bin,credit_score,credit_score_bin,transaction_count,transaction_count_kbin
0,CUST0001,62000.0,Medium,710.0,Good,42.0,1
1,CUST0002,88000.0,Medium,735.0,Good,67.0,2
2,CUST0003,54000.0,Medium,668.0,Fair,36.0,0
3,CUST0004,74000.0,Medium,602.0,Fair,21.0,0
4,CUST0005,145000.0,High,805.0,Excellent,89.0,2


## 16. Binarization

**Intuition:** Binarization converts numeric values into 0/1 flags. This is useful when we want to capture whether a customer crosses a risk threshold.

In [76]:
binarizer = Binarizer(threshold=0.40)
df_encoded['high_debt_to_income_flag'] = binarizer.fit_transform(df_encoded[['debt_to_income_ratio']]).astype(int)
df_encoded['low_credit_score_flag'] = (df_encoded['credit_score'] < 650).astype(int)

display(df_encoded[[id_col, 'debt_to_income_ratio', 'high_debt_to_income_flag', 'credit_score', 'low_credit_score_flag']].head())

,customer_id,debt_to_income_ratio,high_debt_to_income_flag,credit_score,low_credit_score_flag
0,CUST0001,0.241935,0,710.0,0
1,CUST0002,0.250000,0,735.0,0
2,CUST0003,0.222222,0,668.0,0
3,CUST0004,0.243243,0,602.0,1
4,CUST0005,0.275862,0,805.0,0


## 17. Standard Scaling

**Intuition:** StandardScaler converts each numerical feature to mean 0 and standard deviation 1. It is useful for models that are sensitive to feature scale, such as logistic regression, SVM, KNN, and neural networks.

In [77]:
scale_cols = numeric_cols + engineered_features + date_features

standard_scaler = StandardScaler()
standard_array = standard_scaler.fit_transform(df_encoded[scale_cols])
standard_df = pd.DataFrame(standard_array, columns=[f'{col}_standard' for col in scale_cols], index=df_encoded.index)

display(standard_df.head())

,age_standard,annual_income_standard,loan_amount_standard,credit_score_standard,repayment_history_standard,transaction_count_standard,spending_ratio_standard,debt_to_income_ratio_standard,avg_monthly_transactions_standard,spending_to_income_ratio_standard,join_year_standard,join_month_standard,join_day_standard,join_weekday_standard,customer_tenure_days_standard
0,-0.524156,-0.407232,-0.530601,0.330164,-1.130429,-0.426006,-0.760140,-0.532430,-0.426006,-0.868282,0.379762,-0.979587,-0.119276,1.285149,-0.248178
1,-1.072293,0.285079,0.028926,0.678989,-0.585644,0.492605,-0.255748,-0.406916,0.492605,0.390834,-0.080556,1.267494,-1.498849,-1.557577,-0.078670
2,0.480762,-0.620251,-0.770398,-0.255864,-0.040859,-0.646472,-0.504180,-0.839242,-0.646472,-0.883379,-0.540873,-0.136931,1.145334,-1.083789,0.558269
3,-0.067375,-0.087704,-0.290804,-1.176764,1.048712,-1.197639,0.737979,-0.512077,-1.197639,0.937362,0.840080,-1.541357,-0.579134,-1.557577,-0.635992
4,1.120255,1.802838,1.467708,1.655701,-1.130429,1.300982,-1.166665,-0.004406,1.300982,0.432655,-1.001191,0.424839,0.455547,1.285149,0.956355


## 18. Normalization

**Intuition:** Normalization scales each row/vector to unit length. It is useful when direction matters more than magnitude, such as distance-based models.

In [78]:
normalizer = Normalizer(norm='l2')
normalized_array = normalizer.fit_transform(df_encoded[scale_cols])
normalized_df = pd.DataFrame(normalized_array, columns=[f'{col}_normalized' for col in scale_cols], index=df_encoded.index)

display(normalized_df.head())

,age_normalized,annual_income_normalized,loan_amount_normalized,credit_score_normalized,repayment_history_normalized,transaction_count_normalized,spending_ratio_normalized,debt_to_income_ratio_normalized,avg_monthly_transactions_normalized,spending_to_income_ratio_normalized,join_year_normalized,join_month_normalized,join_day_normalized,join_weekday_normalized,customer_tenure_days_normalized
0,0.000518,0.944240,0.228445,0.010813,0.000000,0.000640,0.000378,0.000004,0.000107,0.234171,0.030779,0.000046,0.000213,0.000091,0.017971
1,0.000295,0.927460,0.231865,0.007746,0.000011,0.000706,0.000332,0.000003,0.000118,0.292150,0.021289,0.000116,0.000021,0.000000,0.013828
2,0.000783,0.940064,0.208903,0.011629,0.000035,0.000627,0.000491,0.000004,0.000104,0.265098,0.035148,0.000104,0.000435,0.000017,0.031475
3,0.000470,0.890895,0.216704,0.007248,0.000048,0.000253,0.000538,0.000003,0.000042,0.398230,0.024343,0.000012,0.000120,0.000000,0.010570
4,0.000340,0.947378,0.261346,0.005260,0.000000,0.000581,0.000127,0.000002,0.000097,0.183791,0.013185,0.000052,0.000124,0.000039,0.013838


## 19. Min-Max Scaling

**Intuition:** Min-Max scaling transforms values to the 0 to 1 range. It is good when we want bounded features and know outliers are already handled.

In [79]:
minmax_scaler = MinMaxScaler()
minmax_array = minmax_scaler.fit_transform(df_encoded[scale_cols])
minmax_df = pd.DataFrame(minmax_array, columns=[f'{col}_minmax' for col in scale_cols], index=df_encoded.index)

display(minmax_df.head())

,age_minmax,annual_income_minmax,loan_amount_minmax,credit_score_minmax,repayment_history_minmax,transaction_count_minmax,spending_ratio_minmax,debt_to_income_ratio_minmax,avg_monthly_transactions_minmax,spending_to_income_ratio_minmax,join_year_minmax,join_month_minmax,join_day_minmax,join_weekday_minmax,customer_tenure_days_minmax
0,0.285714,0.308231,0.215343,0.566667,0.000000,0.288210,0.204534,0.406238,0.288210,0.193408,0.625,0.181818,0.433333,1.000000,0.389182
1,0.142857,0.490368,0.366083,0.650000,0.166667,0.506550,0.333816,0.433801,0.506550,0.525641,0.500,0.909091,0.033333,0.000000,0.432718
2,0.547619,0.252189,0.150740,0.426667,0.333333,0.235808,0.270140,0.338862,0.235808,0.189425,0.375,0.454545,0.800000,0.166667,0.596306
3,0.404762,0.392294,0.279946,0.206667,0.666667,0.104803,0.588519,0.410708,0.104803,0.669849,0.750,0.000000,0.300000,0.000000,0.289578
4,0.714286,0.889667,0.753701,0.883333,0.000000,0.698690,0.100338,0.522192,0.698690,0.536676,0.250,0.636364,0.600000,1.000000,0.698549


## 20. MaxAbs Scaling

**Intuition:** MaxAbs scaling divides each feature by its maximum absolute value. It preserves zero values and is commonly useful for sparse data.

In [80]:
maxabs_scaler = MaxAbsScaler()
maxabs_array = maxabs_scaler.fit_transform(df_encoded[scale_cols])
maxabs_df = pd.DataFrame(maxabs_array, columns=[f'{col}_maxabs' for col in scale_cols], index=df_encoded.index)

display(maxabs_df.head())

,age_maxabs,annual_income_maxabs,loan_amount_maxabs,credit_score_maxabs,repayment_history_maxabs,transaction_count_maxabs,spending_ratio_maxabs,debt_to_income_ratio_maxabs,avg_monthly_transactions_maxabs,spending_to_income_ratio_maxabs,join_year_maxabs,join_month_maxabs,join_day_maxabs,join_weekday_maxabs,customer_tenure_days_maxabs
0,0.531250,0.385692,0.291616,0.845238,0.000000,0.340081,0.375615,0.582048,0.340081,0.339092,0.998518,0.250000,0.451613,1.000000,0.389182
1,0.437500,0.547434,0.427704,0.875000,0.166667,0.542510,0.477092,0.601450,0.542510,0.611318,0.998024,0.916667,0.064516,0.000000,0.432718
2,0.703125,0.335925,0.233293,0.795238,0.333333,0.291498,0.427111,0.534622,0.291498,0.335828,0.997530,0.500000,0.806452,0.166667,0.596306
3,0.609375,0.460342,0.349939,0.716667,0.666667,0.170040,0.677016,0.585194,0.170040,0.729480,0.999012,0.083333,0.322581,0.000000,0.289578
4,0.812500,0.902022,0.777643,0.958333,0.000000,0.720648,0.293828,0.663669,0.720648,0.620360,0.997036,0.666667,0.612903,1.000000,0.698549


## 21. Robust Scaling

**Intuition:** Robust scaling uses median and interquartile range. It is useful for credit-risk data because income, loan amount, and spending can contain outliers.

In [81]:
robust_scaler = RobustScaler()
robust_array = robust_scaler.fit_transform(df_encoded[scale_cols])
robust_df = pd.DataFrame(robust_array, columns=[f'{col}_robust' for col in scale_cols], index=df_encoded.index)

display(robust_df.head())

,age_robust,annual_income_robust,loan_amount_robust,credit_score_robust,repayment_history_robust,transaction_count_robust,spending_ratio_robust,debt_to_income_ratio_robust,avg_monthly_transactions_robust,spending_to_income_ratio_robust,join_year_robust,join_month_robust,join_day_robust,join_weekday_robust,customer_tenure_days_robust
0,-0.303030,-0.275862,-0.192,0.177778,-1.0,-0.295775,-0.411043,-1.102513e-01,-0.295775,-0.532272,0.333333,-0.56,-0.070175,0.75,-0.151191
1,-0.666667,0.321839,0.256,0.424691,-0.5,0.408451,0.000000,3.117493e-11,0.408451,0.583998,0.000000,0.72,-0.912281,-0.75,-0.035831
2,0.363636,-0.459770,-0.384,-0.237037,0.0,-0.464789,-0.202454,-3.797545e-01,-0.464789,-0.545656,-0.333333,-0.08,0.701754,-0.50,0.397640
3,0.000000,0.000000,0.000,-0.888889,1.0,-0.887324,0.809816,-9.237271e-02,-0.887324,1.068524,0.666667,-0.88,-0.350877,-0.75,-0.415119
4,0.787879,1.632184,1.408,1.116049,-1.0,1.028169,-0.742331,3.535645e-01,1.028169,0.621075,-0.666667,0.24,0.280702,0.75,0.668560


## 22. FunctionTransformer: Log Transform

**Intuition:** Log transformation reduces right skew in large positive values such as income and loan amount. `log1p` is used because it also works safely with zero values.

In [82]:
log_transformer = FunctionTransformer(np.log1p, feature_names_out='one-to-one')
log_cols = ['annual_income', 'loan_amount', 'spending_to_income_ratio']
log_array = np.asarray(log_transformer.fit_transform(df_encoded[log_cols]))
log_df = pd.DataFrame(log_array, columns=[f'{col}_log' for col in log_cols], index=df_encoded.index)

display(log_df.head())

,annual_income_log,loan_amount_log,spending_to_income_ratio_log
0,11.034906,9.615872,9.640628
1,11.385103,9.998843,10.229946
2,10.896758,9.392745,9.630957
3,11.211834,9.798183,10.406654
4,11.884496,10.596660,10.244627


## 23. FunctionTransformer: Reciprocal Transform

**Intuition:** Reciprocal transformation can reduce the effect of large values. A small epsilon is added to prevent division by zero.

In [83]:
reciprocal_transformer = FunctionTransformer(lambda x: 1 / (x + 1e-6), feature_names_out='one-to-one')
reciprocal_cols = ['annual_income', 'loan_amount', 'credit_score']
reciprocal_array = np.asarray(reciprocal_transformer.fit_transform(df_encoded[reciprocal_cols]))
reciprocal_df = pd.DataFrame(reciprocal_array, columns=[f'{col}_reciprocal' for col in reciprocal_cols], index=df_encoded.index)

display(reciprocal_df.head())

,annual_income_reciprocal,loan_amount_reciprocal,credit_score_reciprocal
0,0.000016,0.000067,0.001408
1,0.000011,0.000045,0.001361
2,0.000019,0.000083,0.001497
3,0.000014,0.000056,0.001661
4,0.000007,0.000025,0.001242


## 24. FunctionTransformer: Square Root Transform

**Intuition:** Square root transformation is milder than log transformation. It can reduce skew while keeping transformed values easier to interpret.

In [84]:
sqrt_transformer = FunctionTransformer(np.sqrt, feature_names_out='one-to-one')
sqrt_cols = ['annual_income', 'loan_amount', 'transaction_count']
sqrt_array = np.asarray(sqrt_transformer.fit_transform(df_encoded[sqrt_cols]))
sqrt_df = pd.DataFrame(sqrt_array, columns=[f'{col}_sqrt' for col in sqrt_cols], index=df_encoded.index)

display(sqrt_df.head())

,annual_income_sqrt,loan_amount_sqrt,transaction_count_sqrt
0,248.997992,122.474487,6.480741
1,296.647939,148.323970,8.185353
2,232.379001,109.544512,6.000000
3,272.029410,134.164079,4.582576
4,380.788655,200.000000,9.433981


## 25. PowerTransformer: Box-Cox

**Intuition:** Box-Cox makes positive numeric data more Gaussian-like. It requires strictly positive values, which fits our income, loan, score, and transaction columns after imputation.

In [85]:
boxcox_cols = ['annual_income', 'loan_amount', 'credit_score', 'transaction_count']
boxcox_transformer = PowerTransformer(method='box-cox', standardize=True)
boxcox_array = boxcox_transformer.fit_transform(df_encoded[boxcox_cols])
boxcox_df = pd.DataFrame(boxcox_array, columns=[f'{col}_boxcox' for col in boxcox_cols], index=df_encoded.index)

display(boxcox_df.head())

,annual_income_boxcox,loan_amount_boxcox,credit_score_boxcox,transaction_count_boxcox
0,-0.289783,-0.364483,0.333282,-0.323130
1,0.400064,0.287010,0.680753,0.561633
2,-0.531557,-0.733708,-0.252390,-0.564211
3,0.044242,-0.057178,-1.177953,-1.249879
4,1.607982,1.350534,1.649423,1.236012


## 26. PowerTransformer: Yeo-Johnson

**Intuition:** Yeo-Johnson is similar to Box-Cox but can also handle zero and negative values. It is safer for general preprocessing pipelines.

In [86]:
yeojohnson_cols = scale_cols
yeojohnson_transformer = PowerTransformer(method='yeo-johnson', standardize=True)
yeojohnson_array = yeojohnson_transformer.fit_transform(df_encoded[yeojohnson_cols])
yeojohnson_df = pd.DataFrame(yeojohnson_array, columns=[f'{col}_yeojohnson' for col in yeojohnson_cols], index=df_encoded.index)

display(yeojohnson_df.head())

,age_yeojohnson,annual_income_yeojohnson,loan_amount_yeojohnson,credit_score_yeojohnson,repayment_history_yeojohnson,transaction_count_yeojohnson,spending_ratio_yeojohnson,debt_to_income_ratio_yeojohnson,avg_monthly_transactions_yeojohnson,spending_to_income_ratio_yeojohnson,join_year_yeojohnson,join_month_yeojohnson,join_day_yeojohnson,join_weekday_yeojohnson,customer_tenure_days_yeojohnson
0,-0.440131,-0.289784,-0.364486,0.333277,-1.470137,-0.324684,-0.727575,-0.462056,-0.330869,-0.870140,0.365913,-0.955380,-0.021649,1.244268,-0.135179
1,-1.111084,0.400064,0.287011,0.680750,-0.415105,0.561525,-0.058655,-0.304920,0.561094,0.536169,-0.097038,1.217085,-1.647797,-1.629813,0.028121
2,0.568242,-0.531558,-0.733713,-0.252396,0.232325,-0.565952,-0.364729,-0.875288,-0.572810,-0.892018,-0.552998,-0.060822,1.103432,-1.071890,0.602381
3,0.047557,0.044242,-0.057179,-1.177951,1.081327,-1.250662,0.869743,-0.436124,-1.253309,0.987241,0.835960,-1.664378,-0.490103,-1.629813,-0.531464
4,1.106868,1.607982,1.350537,1.649434,-1.470137,1.237141,-1.443232,0.157069,1.241620,0.573111,-1.002068,0.473897,0.513112,1.244268,0.936843


## 27. ColumnTransformer Pipeline

**Intuition:** A ColumnTransformer applies different preprocessing steps to different column groups in one clean pipeline. This is the professional way to prepare data for ML modeling.

In [87]:
pipeline_numeric_cols = scale_cols + ['high_debt_to_income_flag', 'low_credit_score_flag', 'transaction_count_kbin']
pipeline_ordinal_cols = ['education_level']
pipeline_nominal_cols = nominal_cols + ['income_bin', 'credit_score_bin']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=education_order))
])

nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, pipeline_numeric_cols),
    ('ord', ordinal_pipeline, pipeline_ordinal_cols),
    ('nom', nominal_pipeline, pipeline_nominal_cols)
])

X_for_pipeline = df_encoded[pipeline_numeric_cols + pipeline_ordinal_cols + pipeline_nominal_cols]
X_transformed = preprocessor.fit_transform(X_for_pipeline)

feature_names = preprocessor.get_feature_names_out()
pipeline_transformed_df = pd.DataFrame(X_transformed, columns=feature_names, index=df_encoded.index)

print('Pipeline transformed shape:', pipeline_transformed_df.shape)
display(pipeline_transformed_df.head())

Pipeline transformed shape: (80, 43)


,num__age,num__annual_income,num__loan_amount,num__credit_score,num__repayment_history,num__transaction_count,num__spending_ratio,num__debt_to_income_ratio,num__avg_monthly_transactions,num__spending_to_income_ratio,num__join_year,num__join_month,num__join_day,num__join_weekday,num__customer_tenure_days,num__high_debt_to_income_flag,num__low_credit_score_flag,num__transaction_count_kbin,ord__education_level,nom__gender_Female,nom__gender_Male,nom__region_East,nom__region_North,nom__region_South,nom__region_West,nom__employment_type_Retired,nom__employment_type_Salaried,nom__employment_type_Self-Employed,nom__employment_type_Student,nom__employment_type_Unemployed,nom__loan_purpose_Business,nom__loan_purpose_Car,nom__loan_purpose_Education,nom__loan_purpose_Home,nom__loan_purpose_Medical,nom__loan_purpose_Personal,nom__income_bin_High,nom__income_bin_Low,nom__income_bin_Medium,nom__credit_score_bin_Excellent,nom__credit_score_bin_Fair,nom__credit_score_bin_Good,nom__credit_score_bin_Poor
0,-0.303030,-0.275862,-0.192,0.177778,-1.0,-0.295775,-0.411043,-1.102513e-01,-0.295775,-0.532272,0.333333,-0.56,-0.070175,0.75,-0.151191,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-0.666667,0.321839,0.256,0.424691,-0.5,0.408451,0.000000,3.117493e-11,0.408451,0.583998,0.000000,0.72,-0.912281,-0.75,-0.035831,0.0,0.0,0.5,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.363636,-0.459770,-0.384,-0.237037,0.0,-0.464789,-0.202454,-3.797545e-01,-0.464789,-0.545656,-0.333333,-0.08,0.701754,-0.50,0.397640,0.0,0.0,-0.5,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.000000,0.000000,0.000,-0.888889,1.0,-0.887324,0.809816,-9.237271e-02,-0.887324,1.068524,0.666667,-0.88,-0.350877,-0.75,-0.415119,0.0,1.0,-0.5,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.787879,1.632184,1.408,1.116049,-1.0,1.028169,-0.742331,3.535645e-01,1.028169,0.621075,-0.666667,0.24,0.280702,0.75,0.668560,0.0,0.0,0.5,3.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


## 28. Build Final Cleaned and Transformed Dataset

**Intuition:** The final ML-ready dataset should contain the target and transformed numeric features. We remove raw text/date identifiers from the model-ready feature matrix but keep `customer_id` separately for traceability.

In [88]:
final_dataset = pd.concat(
    [
        df_encoded[[id_col, target_col]].reset_index(drop=True),
        pipeline_transformed_df.reset_index(drop=True),
        standard_df.reset_index(drop=True),
        minmax_df.reset_index(drop=True),
        log_df.reset_index(drop=True),
        boxcox_df.reset_index(drop=True),
        yeojohnson_df.reset_index(drop=True)
    ],
    axis=1
)

final_dataset = final_dataset.loc[:, ~final_dataset.columns.duplicated()]

print('Final dataset shape:', final_dataset.shape)
print('Missing values in final dataset:', int(final_dataset.isna().sum().sum()))
display(final_dataset.head())

Final dataset shape: (80, 97)
Missing values in final dataset: 0


,customer_id,default_flag,num__age,num__annual_income,num__loan_amount,num__credit_score,num__repayment_history,num__transaction_count,num__spending_ratio,num__debt_to_income_ratio,num__avg_monthly_transactions,num__spending_to_income_ratio,num__join_year,num__join_month,num__join_day,num__join_weekday,num__customer_tenure_days,num__high_debt_to_income_flag,num__low_credit_score_flag,num__transaction_count_kbin,ord__education_level,nom__gender_Female,nom__gender_Male,nom__region_East,nom__region_North,nom__region_South,nom__region_West,nom__employment_type_Retired,nom__employment_type_Salaried,nom__employment_type_Self-Employed,nom__employment_type_Student,nom__employment_type_Unemployed,nom__loan_purpose_Business,nom__loan_purpose_Car,nom__loan_purpose_Education,nom__loan_purpose_Home,nom__loan_purpose_Medical,nom__loan_purpose_Personal,nom__income_bin_High,nom__income_bin_Low,nom__income_bin_Medium,nom__credit_score_bin_Excellent,nom__credit_score_bin_Fair,nom__credit_score_bin_Good,nom__credit_score_bin_Poor,age_standard,annual_income_standard,loan_amount_standard,credit_score_standard,repayment_history_standard,transaction_count_standard,spending_ratio_standard,debt_to_income_ratio_standard,avg_monthly_transactions_standard,spending_to_income_ratio_standard,join_year_standard,join_month_standard,join_day_standard,join_weekday_standard,customer_tenure_days_standard,age_minmax,annual_income_minmax,loan_amount_minmax,credit_score_minmax,repayment_history_minmax,transaction_count_minmax,spending_ratio_minmax,debt_to_income_ratio_minmax,avg_monthly_transactions_minmax,spending_to_income_ratio_minmax,join_year_minmax,join_month_minmax,join_day_minmax,join_weekday_minmax,customer_tenure_days_minmax,annual_income_log,loan_amount_log,spending_to_income_ratio_log,annual_income_boxcox,loan_amount_boxcox,credit_score_boxcox,transaction_count_boxcox,age_yeojohnson,annual_income_yeojohnson,loan_amount_yeojohnson,credit_score_yeojohnson,repayment_history_yeojohnson,transaction_count_yeojohnson,spending_ratio_yeojohnson,debt_to_income_ratio_yeojohnson,avg_monthly_transactions_yeojohnson,spending_to_income_ratio_yeojohnson,join_year_yeojohnson,join_month_yeojohnson,join_day_yeojohnson,join_weekday_yeojohnson,customer_tenure_days_yeojohnson
0,CUST0001,0,-0.303030,-0.275862,-0.192,0.177778,-1.0,-0.295775,-0.411043,-1.102513e-01,-0.295775,-0.532272,0.333333,-0.56,-0.070175,0.75,-0.151191,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,-0.524156,-0.407232,-0.530601,0.330164,-1.130429,-0.426006,-0.760140,-0.532430,-0.426006,-0.868282,0.379762,-0.979587,-0.119276,1.285149,-0.248178,0.285714,0.308231,0.215343,0.566667,0.000000,0.288210,0.204534,0.406238,0.288210,0.193408,0.625,0.181818,0.433333,1.000000,0.389182,11.034906,9.615872,9.640628,-0.289783,-0.364483,0.333282,-0.323130,-0.440131,-0.289784,-0.364486,0.333277,-1.470137,-0.324684,-0.727575,-0.462056,-0.330869,-0.870140,0.365913,-0.955380,-0.021649,1.244268,-0.135179
1,CUST0002,0,-0.666667,0.321839,0.256,0.424691,-0.5,0.408451,0.000000,3.117493e-11,0.408451,0.583998,0.000000,0.72,-0.912281,-0.75,-0.035831,0.0,0.0,0.5,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,-1.072293,0.285079,0.028926,0.678989,-0.585644,0.492605,-0.255748,-0.406916,0.492605,0.390834,-0.080556,1.267494,-1.498849,-1.557577,-0.078670,0.142857,0.490368,0.366083,0.650000,0.166667,0.506550,0.333816,0.433801,0.506550,0.525641,0.500,0.909091,0.033333,0.000000,0.432718,11.385103,9.998843,10.229946,0.400064,0.287010,0.680753,0.561633,-1.111084,0.400064,0.287011,0.680750,-0.415105,0.561525,-0.058655,-0.304920,0.561094,0.536169,-0.097038,1.217085,-1.647797,-1.629813,0.028121
2,CUST0003,0,0.363636,-0.459770,-0.384,-0.237037,0.0,-0.464789,-0.202454,-3.797545e-01,-0.464789,-0.545656,-0.333333,-0.08,0.701754,-0.50,0.397640,0.0,0.0,-0.5,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0

## 29. Save Final Dataset

**Intuition:** Saving the final dataset creates the main deliverable. This file can be used directly for ML model training.

In [89]:
output_file = 'final_cleaned_transformed_customer_credit_risk_dataset.csv'
final_dataset.to_csv(output_file, index=False)

print(f'Final cleaned and transformed dataset saved as: {output_file}')

Final cleaned and transformed dataset saved as: final_cleaned_transformed_customer_credit_risk_dataset.csv


## 30. Final Report

**Intuition:** The report explains what was done and why. This is useful for project submission because preprocessing decisions must be justified, not only coded.

In [90]:
report = f'''
FINAL PREPROCESSING REPORT
==========================

1. Dataset Used
- Source file: customer_credit_risk_practice_dataset.csv
- Original shape: {raw_df.shape}
- Final transformed shape: {final_dataset.shape}

2. Missing Value Strategies
- Numerical columns were imputed using median values because median is robust to outliers.
- Categorical columns were imputed using most frequent values because categories are discrete.
- join_date was imputed using the median date so date-derived features could be created.
- KNN imputation was demonstrated separately for numerical practice.

3. Outlier Handling
- IQR method was used to detect outliers.
- Capping/Winsorization was used instead of row deletion to preserve dataset size.

4. Encoding Methods
- Ordinal encoding was used for education_level because it has a natural order.
- One-hot encoding was used for gender, region, employment_type, loan_purpose, income_bin, and credit_score_bin.
- Label encoding was demonstrated for default_flag, although it was already binary.

5. Scaling and Transformations
- Standard Scaling, Normalization, Min-Max Scaling, MaxAbs Scaling, and Robust Scaling were applied separately.
- RobustScaler was selected inside the final pipeline because credit-risk data contains outliers.
- FunctionTransformer was used for log, reciprocal, and square-root transformations.
- PowerTransformer was used for Box-Cox and Yeo-Johnson transformations.

6. Feature Engineering
- debt_to_income_ratio: measures loan burden compared with annual income.
- avg_monthly_transactions: estimates average monthly transaction activity from six-month count.
- spending_to_income_ratio: estimates spending burden in relation to income.
- Date features: join_year, join_month, join_day, join_weekday, and customer_tenure_days.

7. Readiness for ML Modeling
- Final dataset contains no missing values.
- Categorical variables are numerically encoded.
- Numerical variables are scaled/transformed.
- Engineered risk features are included.
- The dataset is ready for model training and evaluation.
'''

print(report)

with open('preprocessing_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print('Report saved as: preprocessing_report.txt')


FINAL PREPROCESSING REPORT

1. Dataset Used
- Source file: customer_credit_risk_practice_dataset.csv
- Original shape: (80, 15)
- Final transformed shape: (80, 97)

2. Missing Value Strategies
- Numerical columns were imputed using median values because median is robust to outliers.
- Categorical columns were imputed using most frequent values because categories are discrete.
- join_date was imputed using the median date so date-derived features could be created.
- KNN imputation was demonstrated separately for numerical practice.

3. Outlier Handling
- IQR method was used to detect outliers.
- Capping/Winsorization was used instead of row deletion to preserve dataset size.

4. Encoding Methods
- Ordinal encoding was used for education_level because it has a natural order.
- One-hot encoding was used for gender, region, employment_type, loan_purpose, income_bin, and credit_score_bin.
- Label encoding was demonstrated for default_flag, although it was already binary.

5. Scaling and Tr

## 31. Final Requirements Checklist

**Completed tasks:**

- Loaded and inspected the customer credit risk dataset.
- Handled missing values using median, mode, date median, and KNN demonstration.
- Detected outliers using IQR.
- Handled outliers using capping/Winsorization.
- Encoded ordinal and nominal categorical variables.
- Demonstrated label encoding for the binary target.
- Applied binning, discretization, and binarization.
- Applied Standard Scaling, Normalization, Min-Max Scaling, MaxAbs Scaling, and Robust Scaling.
- Applied FunctionTransformer using log, reciprocal, and square-root transforms.
- Applied PowerTransformer using Box-Cox and Yeo-Johnson.
- Used ColumnTransformer to apply different preprocessing steps to different columns.
- Constructed debt-to-income ratio, average monthly transactions, and spending-to-income ratio.
- Created a final cleaned and transformed dataset.
- Generated a final preprocessing report.